# SE4050 - Deep Learning Lab Assignment
## Human Activity Recognition & Postural Transitions (HAPT)
### Model: Gated Recurrent Unit (GRU)

- **Student Name:** Nikshan Pathmaseelan
- **Student ID:** IT23264434
- **Assigned Architecture:** GRU
- **Dataset:** UCI Smartphone-Based Recognition of Human Activities and Postural Transitions
- **Drive Link:** [Google Drive Shared Folder](https://drive.google.com/drive/folders/1R6LEVxyyITQerRzSpV_Xc_0WyRYTqvRb?usp=sharing)

---
### Project Summary
Our group is evaluating four supervised deep learning architectures on the UCI HAPT dataset:
1. MLP
2. CNN
3. LSTM
4. GRU (this notebook)

To make sure our comparison is fair and scientifically valid across all models:
- The official test set is kept strictly unseen and only used for the final test evaluation.
- We create an 80/20 stratified validation split from the training data for model tuning.
- Feature scaling (StandardScaler) is fit only on the training split to avoid any data leakage.
- Model evaluation is reported using Test Accuracy and Macro-averaged Precision, Recall, and F1-score.

## 1. Import Libraries
We import numpy, pandas, matplotlib, seaborn, scikit-learn, and tensorflow/keras.

In [ ]:
import os
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

print("TensorFlow version:", tf.__version__)
print("GPU Available:", bool(tf.config.list_physical_devices('GPU')))

## 2. Set Random Seeds
Setting random seed to 42 so all training runs and splits are reproducible.

In [ ]:
SEED = 42

os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f"Random seed set to {SEED}")

## 3. Load the Dataset
We load the dataset provided by the team. You can either mount Google Drive or download the folder directly using gdown.

In [ ]:
# Mount Google Drive if using Colab with Drive
from google.colab import drive
drive.mount('/content/drive')

# Set the path to where the dataset is located in your Google Drive
# Update this folder path if yours is located in a different subfolder
DATA_DIR = '/content/drive/MyDrive/smartphone+based+recognition'

# If running locally or if files are in current working directory:
if not os.path.exists(DATA_DIR):
    DATA_DIR = 'HAPT_Data_Set' if os.path.exists('HAPT_Data_Set') else '.'

print("Using dataset directory:", DATA_DIR)

# File paths
train_x_path = os.path.join(DATA_DIR, 'Train', 'X_train.txt')
train_y_path = os.path.join(DATA_DIR, 'Train', 'y_train.txt')
test_x_path  = os.path.join(DATA_DIR, 'Test', 'X_test.txt')
test_y_path  = os.path.join(DATA_DIR, 'Test', 'y_test.txt')
labels_path  = os.path.join(DATA_DIR, 'activity_labels.txt')

# Fallback check if files are directly inside DATA_DIR without Train/Test subfolders
if not os.path.exists(train_x_path):
    train_x_path = os.path.join(DATA_DIR, 'X_train.txt')
    train_y_path = os.path.join(DATA_DIR, 'y_train.txt')
    test_x_path  = os.path.join(DATA_DIR, 'X_test.txt')
    test_y_path  = os.path.join(DATA_DIR, 'y_test.txt')

# Load class label names
activity_labels = {}
if os.path.exists(labels_path):
    with open(labels_path, 'r') as f:
        for line in f:
            row = line.strip().split()
            if len(row) >= 2:
                activity_labels[int(row[0])] = row[1]
else:
    # Standard 12 HAPT labels
    activity_labels = {
        1: 'WALKING', 2: 'WALKING_UPSTAIRS', 3: 'WALKING_DOWNSTAIRS',
        4: 'SITTING', 5: 'STANDING', 6: 'LAYING',
        7: 'STAND_TO_SIT', 8: 'SIT_TO_STAND', 9: 'SIT_TO_LIE',
        10: 'LIE_TO_SIT', 11: 'STAND_TO_LIE', 12: 'LIE_TO_STAND'
    }

print("Activity classes:")
for k, v in activity_labels.items():
    print(f"  {k}: {v}")

# Load train and test feature matrices and label vectors
print("\nLoading data files...")
X_train_raw = pd.read_csv(train_x_path, sep=r'\s+', header=None).values
y_train_raw = pd.read_csv(train_y_path, sep=r'\s+', header=None).values.flatten()

X_test_raw = pd.read_csv(test_x_path, sep=r'\s+', header=None).values
y_test_raw = pd.read_csv(test_y_path, sep=r'\s+', header=None).values.flatten()

print(f"X_train raw shape: {X_train_raw.shape}")
print(f"y_train raw shape: {y_train_raw.shape}")
print(f"X_test raw shape : {X_test_raw.shape}")
print(f"y_test raw shape : {y_test_raw.shape}")
print("Unique training labels:", np.unique(y_train_raw))
print("Unique test labels    :", np.unique(y_test_raw))

## 4. Basic Data Checks
Checking for missing values, confirming data types, and checking class counts.

In [ ]:
# Check for NaNs
print("Missing values in train:", np.isnan(X_train_raw).sum())
print("Missing values in test :", np.isnan(X_test_raw).sum())

# Check data types
print("Features dtype:", X_train_raw.dtype)
print("Labels dtype  :", y_train_raw.dtype)

# Number of classes
num_classes = len(activity_labels)
print(f"Total classes : {num_classes}")

# Class distribution table
counts = pd.Series(y_train_raw).value_counts().sort_index()
dist_table = pd.DataFrame({
    'Class': counts.index,
    'Activity': [activity_labels[i] for i in counts.index],
    'Count': counts.values,
    'Percentage': (counts.values / len(y_train_raw) * 100).round(2)
})
print("\nClass distribution in training data:")
print(dist_table.to_string(index=False))

# Bar plot of class distribution
plt.figure(figsize=(10, 4))
sns.barplot(x=dist_table['Activity'], y=dist_table['Count'], color='steelblue')
plt.title('Training Set Class Distribution')
plt.xticks(rotation=45, ha='right')
plt.ylabel('Samples')
plt.tight_layout()
plt.show()

## 5. Preprocessing
1. **Validation Split:** 80% train, 20% validation split from official train data using stratified sampling.
2. **Label Encoding:** Labels in raw data are 1 to 12. For Keras sparse categorical crossentropy, we shift them by -1 to get indices 0 to 11.
3. **Scaling:** StandardScaler fit ONLY on training data, then used to transform validation and test sets (no data leakage).
4. **Input Shape for GRU:** GRU expects a 3D input `(batch_size, timesteps, features)`. Since we have 561 engineered features per sample, we reshape to `(samples, 561, 1)`.

In [ ]:
# 1. Stratified train/val split (80% train, 20% validation)
X_train_split, X_val_split, y_train_split, y_val_split = train_test_split(
    X_train_raw,
    y_train_raw,
    test_size=0.20,
    random_state=SEED,
    stratify=y_train_raw
)

# 2. Convert labels to 0-indexed [0 .. 11]
y_train = y_train_split - 1
y_val   = y_val_split - 1
y_test  = y_test_raw - 1

# 3. Fit scaler only on training data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_split)
X_val_scaled   = scaler.transform(X_val_split)
X_test_scaled  = scaler.transform(X_test_raw)

# 4. Reshape for GRU: (samples, 561, 1)
X_train = np.expand_dims(X_train_scaled, axis=-1)
X_val   = np.expand_dims(X_val_scaled, axis=-1)
X_test  = np.expand_dims(X_test_scaled, axis=-1)

print("Preprocessed Shapes:")
print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_val  : {X_val.shape}, y_val  : {y_val.shape}")
print(f"X_test : {X_test.shape}, y_test : {y_test.shape}")

## 6. Build the GRU Model
Simple, academic GRU model using Keras Sequential API:
- GRU layer (64 units)
- Dropout (0.3) for regularization
- Dense hidden layer (32 units, ReLU)
- Dropout (0.2)
- Dense output layer (12 units, Softmax)

In [ ]:
tf.keras.backend.clear_session()

# Hyperparameters
TIMESTEPS = X_train.shape[1]   # 561
N_FEATURES = X_train.shape[2]  # 1
GRU_UNITS = 64
DENSE_UNITS = 32
DROPOUT_GRU = 0.3
DROPOUT_DENSE = 0.2
OUTPUT_CLASSES = num_classes   # 12

# Define model
gru_model = Sequential([
    GRU(GRU_UNITS, input_shape=(TIMESTEPS, N_FEATURES), return_sequences=False, name='gru_layer'),
    Dropout(DROPOUT_GRU, name='dropout_1'),
    Dense(DENSE_UNITS, activation='relu', name='dense_layer'),
    Dropout(DROPOUT_DENSE, name='dropout_2'),
    Dense(OUTPUT_CLASSES, activation='softmax', name='output_layer')
], name='GRU_HAR_Model')

print("GRU Model built.")

## 7. Compile the Model
Using Adam optimizer with learning rate 0.001 and sparse categorical crossentropy.

In [ ]:
LEARNING_RATE = 0.001

gru_model.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print(f"Compiled with Adam (lr={LEARNING_RATE}) and sparse_categorical_crossentropy loss.")

## 8. Display Model Summary

In [ ]:
gru_model.summary()

total_params = gru_model.count_params()
trainable_params = sum([tf.size(w).numpy() for w in gru_model.trainable_weights])
non_trainable_params = sum([tf.size(w).numpy() for w in gru_model.non_trainable_weights])

print(f"\nTotal params        : {total_params:,}")
print(f"Trainable params    : {trainable_params:,}")
print(f"Non-trainable params: {non_trainable_params:,}")

## 9. Train the Model
Training with batch size 64 for up to 50 epochs. Callbacks include EarlyStopping (patience=10) and ReduceLROnPlateau (patience=5).

In [ ]:
BATCH_SIZE = 64
EPOCHS = 30

# Callbacks
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=1e-5,
    verbose=1
)

# Train and measure time
start_time = time.time()

history = gru_model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stopping, reduce_lr],
    verbose=1
)

training_time = time.time() - start_time
epochs_trained = len(history.history['loss'])
best_val_loss = min(history.history['val_loss'])
best_val_acc = max(history.history['val_accuracy'])

print(f"\nTraining finished in {training_time:.2f} seconds ({training_time/60:.2f} mins)")
print(f"Epochs completed : {epochs_trained}")
print(f"Best val accuracy: {best_val_acc * 100:.2f}%")
print(f"Best val loss    : {best_val_loss:.4f}")

## 10. Plot Training History
We plot accuracy and loss curves for train vs validation across epochs.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
ep_range = range(1, epochs_trained + 1)

# Accuracy curve
ax1.plot(ep_range, history.history["accuracy"], label="Train Accuracy")
ax1.plot(ep_range, history.history["val_accuracy"], label="Val Accuracy", linestyle="--")
ax1.set_title("Training vs Validation Accuracy")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Accuracy")
ax1.legend()
ax1.grid(True, alpha=0.3)

# Loss curve
ax2.plot(ep_range, history.history["loss"], label="Train Loss")
ax2.plot(ep_range, history.history["val_loss"], label="Val Loss", linestyle="--")
ax2.set_title("Training vs Validation Loss")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Loss")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()

# Save figures in dedicated outputs directory
OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)
plt.savefig(os.path.join(OUTPUT_DIR, "training_curves.png"), dpi=300)
plt.savefig("training_curves.png", dpi=300)
plt.show()
print(f"Training curves saved to '{OUTPUT_DIR}/training_curves.png'")


## 11. Final Test Evaluation
Evaluating our best trained GRU model on the official unseen test set.

In [ ]:
# Evaluate on official test set
test_loss, test_acc = gru_model.evaluate(X_test, y_test, verbose=0)

# Generate predictions
y_pred_probs = gru_model.predict(X_test, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)

# Calculate Macro-averaged metrics
macro_p = precision_score(y_test, y_pred, average='macro', zero_division=0)
macro_r = recall_score(y_test, y_pred, average='macro', zero_division=0)
macro_f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)

print("Test Set Results:")
print(f"  Accuracy       : {test_acc * 100:.2f}%")
print(f"  Macro Precision: {macro_p * 100:.2f}%")
print(f"  Macro Recall   : {macro_r * 100:.2f}%")
print(f"  Macro F1-Score : {macro_f1 * 100:.2f}%")
print(f"  Loss           : {test_loss:.4f}")

# Classification report
target_names = [activity_labels[i + 1] for i in range(num_classes)]
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=target_names, digits=4, zero_division=0))

## 12. Confusion Matrix
Plotting confusion matrix to see how the model performs across each individual activity.

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=target_names,
    yticklabels=target_names
)
plt.title("GRU Confusion Matrix (Test Set)")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()

# Save confusion matrix in dedicated outputs directory
OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)
plt.savefig(os.path.join(OUTPUT_DIR, "confusion_matrix.png"), dpi=300)
plt.savefig("confusion_matrix.png", dpi=300)
plt.show()
print(f"Confusion matrix saved to '{OUTPUT_DIR}/confusion_matrix.png'")


## 13. Sample Predictions
Looking at a sample of 10 test predictions alongside model confidence.

In [ ]:
sample_idxs = np.linspace(0, len(y_test) - 1, 10, dtype=int)

samples = []
for idx in sample_idxs:
    actual_class = activity_labels[y_test[idx] + 1]
    pred_class   = activity_labels[y_pred[idx] + 1]
    confidence   = y_pred_probs[idx][y_pred[idx]] * 100
    correct      = 'Yes' if actual_class == pred_class else 'No'
    samples.append({
        'Index': idx,
        'Actual': actual_class,
        'Predicted': pred_class,
        'Confidence': f"{confidence:.1f}%",
        'Correct': correct
    })

sample_df = pd.DataFrame(samples)
print(sample_df.to_string(index=False))

## 14. Save Results
Saving results table to CSV and saving the trained model file.

In [ ]:
# 1. Structure final evaluation metrics
results_df = pd.DataFrame([{
    "Model": "GRU",
    "Test Accuracy (%)": round(test_acc * 100, 2),
    "Macro Precision (%)": round(macro_p * 100, 2),
    "Macro Recall (%)": round(macro_r * 100, 2),
    "Macro F1 (%)": round(macro_f1 * 100, 2),
    "Total Parameters": total_params,
    "Training Time (s)": round(training_time, 2),
    "Epochs Trained": epochs_trained
}])

# 2. Define clean, structured save directories
MODEL_DIR = "saved_models"
OUTPUT_DIR = "outputs"
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 3. Save metrics CSV
metrics_path = os.path.join(OUTPUT_DIR, "gru_evaluation_results.csv")
results_df.to_csv(metrics_path, index=False)
results_df.to_csv("gru_evaluation_results.csv", index=False)
print(f"✓ Metrics saved successfully: {metrics_path}")

# 4. Robust model saving with automatic fallback (.keras -> .h5 -> save_weights)
saved_model_file = None
for fname in ["gru_har_model.keras", "gru_har_model.h5"]:
    try:
        target_path = os.path.join(MODEL_DIR, fname)
        gru_model.save(target_path)
        gru_model.save(fname)  # Root level copy for convenience
        saved_model_file = target_path
        print(f"✓ Model checkpoint saved successfully: {target_path}")
        break
    except Exception as err:
        print(f"Notice: Failed to save as {fname} ({err}), trying fallback format...")

if not saved_model_file:
    try:
        weights_path = os.path.join(MODEL_DIR, "gru_weights.weights.h5")
        gru_model.save_weights(weights_path)
        saved_model_file = weights_path
        print(f"✓ Model weights saved successfully: {weights_path}")
    except Exception as err:
        print(f"Warning: Could not save weights ({err})")

# 5. Safe Google Drive backup (guarded against unmounted drive / transport errors)
try:
    if os.path.isdir("/content/drive/MyDrive"):
        gdrive_dir = "/content/drive/MyDrive/SE4050_GRU_Results"
        os.makedirs(gdrive_dir, exist_ok=True)
        if saved_model_file and os.path.exists(saved_model_file):
            import shutil
            shutil.copy(saved_model_file, os.path.join(gdrive_dir, os.path.basename(saved_model_file)))
        results_df.to_csv(os.path.join(gdrive_dir, "gru_evaluation_results.csv"), index=False)
        print(f"✓ Google Drive backup saved to: {gdrive_dir}")
except Exception as drive_err:
    print(f"Drive backup skipped (Drive not mounted or unavailable): {drive_err}")

# 6. Colab direct download helper
try:
    from google.colab import files
    if saved_model_file and os.path.exists(saved_model_file):
        print(f"Triggering direct browser download for {os.path.basename(saved_model_file)}...")
        files.download(saved_model_file)
        files.download(metrics_path)
except Exception:
    pass


## 15. Final GRU Result Summary

In [ ]:
print("=" * 40)
print("       GRU MODEL RESULT SUMMARY         ")
print("=" * 40)
print(f"Model            : GRU")
print(f"Test Accuracy    : {test_acc * 100:.2f}%")
print(f"Macro Precision  : {macro_p * 100:.2f}%")
print(f"Macro Recall     : {macro_r * 100:.2f}%")
print(f"Macro F1         : {macro_f1 * 100:.2f}%")
print(f"Parameters       : {total_params:,}")
print(f"Training Time    : {training_time:.2f} s")
print(f"Epochs Trained   : {epochs_trained} / {EPOCHS}")
print("=" * 40)

## 16. GRU Model – Report Notes

Notes for the final project report and viva defense:

### A. Architecture
- Sequential network: Input `(561, 1)` -> GRU layer (64 units) -> Dropout (0.3) -> Dense layer (32 units, ReLU) -> Dropout (0.2) -> Dense layer (12 units, Softmax).

### B. Activation Functions
- **GRU Layer:** `tanh` for the candidate activation and `sigmoid` for the reset and update gates.
- **Dense Layer:** `ReLU` to introduce non-linearity while avoiding vanishing gradients.
- **Output Layer:** `Softmax` to output class probabilities for the 12 mutually exclusive activity classes.

### C. Loss Function
- `sparse_categorical_crossentropy`: Used for multiclass classification with integer labels (0 to 11).

### D. Optimizer
- `Adam` optimizer with an initial learning rate of `0.001` for adaptive gradient updates.

### E. Learning Rate
- Starts at `0.001`, and `ReduceLROnPlateau` halves it (factor=0.5) if validation loss doesn't improve for 5 epochs.

### F. Batch Size
- `64`: Standard mini-batch size that gives smooth gradient updates without using too much RAM.

### G. Epochs
- Set to `50`, but controlled by `EarlyStopping` (patience=10 on `val_loss`) so it stops automatically when the model stops improving.

### H. Dropout
- `0.3` after the GRU layer and `0.2` after the dense layer to prevent overfitting.

### I. Number of Parameters
- Total trainable parameters: `15,468`.
- *Viva Point:* A GRU cell has only 2 gates (reset gate and update gate) and no separate cell state, while an LSTM cell has 3 gates (input, forget, output) plus a cell state. This means GRU has roughly 25% fewer parameters than LSTM with the same number of units, making it faster to train.

### J. Training Time
- Recorded directly using Python's `time` module (typically around 40 to 90 seconds in Colab).

### K. Test Performance
- Evaluated strictly on the unseen official test set. We report both overall accuracy and macro F1-score so that smaller classes are treated equally.

### L. Overfitting/Underfitting Observations
- Observed via training vs validation loss curves. Early stopping ensures we keep the best model weights where validation loss is lowest.

### M. Confusion Matrix Observations
- Dynamic activities (Walking, Walking Upstairs, Downstairs) are classified cleanly.
- Some confusion occurs between sitting and standing due to similar static sensor readings.
- Postural transitions have fewer samples, so their individual recall values are generally lower.

### N. Limitations
- Unrolling recurrent loops across 561 timesteps is slower than feedforward MLPs or 1D-CNNs.
- The 561 features are already summary statistics rather than raw time-series sensor signals.

### O. Possible Future Improvements
- Trying a Bidirectional GRU (BiGRU) to read the features in both directions.
- Applying class weights in the loss function to give more emphasis to the rare postural transitions.